# Module 32 — Graph Engineering & Temporal Knowledge

Predict → Build → Try → Break → Debug → Measure → Improve → Defend

Goal: make graph knowledge historically queryable, versioned, provenance-backed and safe to evolve.

## Temporal concept map
Valid time = when a fact is true. Transaction time = when our system recorded the belief. Observation/ingestion time describe the source/reporting lifecycle.

Never collapse these into one timestamp when historical audit matters.

In [ ]:
from datetime import date
from app.temporal import TemporalEdge, detect_overlap
from app.temporal_validation import Interval, overlaps, reject_overlapping_exclusive
a=TemporalEdge('SupplierA','OWNS','Payments',date(2025,1,1),date(2025,6,30),'contract-v3')
b=TemporalEdge('SupplierA','OWNS','Payments',date(2025,7,1),None,'contract-v4')
print('BUILD current=', b.active_on(date(2025,8,1)))
print('BUILD historical=', a.active_on(date(2025,3,1)))

## TRY — late-arriving correction
A source received in March can report a fact that became true in January. Keep source/transaction timing distinct from valid time.

In [ ]:
late=TemporalEdge('SupplierA','OWNS','Identity',date(2026,1,1),date(2026,2,15),'late-source')
print('TRY late correction valid=', late.active_on(date(2026,2,1)))

## BREAK — overlapping exclusive intervals
Two exclusive ownership intervals overlap. The validator should fail closed.

In [ ]:
i1=Interval(date(2025,1,1),date(2025,6,30))
i2=Interval(date(2025,6,1),None)
print('BREAK overlap detected=', overlaps(i1,i2))
try:
    reject_overlapping_exclusive([i1,i2])
except ValueError as exc:
    print('BREAK/DEBUG:', exc)

## BREAK — graph-level overlap
The existing graph fixture can also detect overlapping edges with the same semantic key.

In [ ]:
bad=TemporalEdge('SupplierA','OWNS','Payments',date(2025,6,1),None,'bad-source')
print('BREAK/DEBUG graph overlaps=', len(detect_overlap([a,bad])))

## Schema evolution challenge
Design v1 → v2 migration rules for a relation rename or cardinality change. Include dual-read/dual-write, backfill, validation, rollback and compatibility tests.

## MEASURE
Track provenance coverage, temporal-overlap rate, stale-edge rate, contradiction rate, entity-resolution precision, traversal work, p95 latency and migration error rate.

In [ ]:
edges=[a,b]
coverage=sum(bool(e.provenance) for e in edges)/len(edges)
print('MEASURE provenance_coverage=', coverage)
print('MEASURE overlap_rate=', len(detect_overlap(edges))/1)

## Domain labs
Cybersecurity: reconstruct controls valid on an incident date.
Banking: identify the policy version governing a transaction date.
Healthcare: prevent expired guidance from being treated as current.
Manufacturing: detect overlapping component ownership.
Enterprise IT: reconstruct dependency state during an outage.

## DEFEND / mastery gate
Show temporal correctness, provenance, tenant/ACL controls, bounded traversal, schema migration and rollback evidence. Explain why supersession is not the same as contradiction.

**Handoff:** M32 provides durable temporal semantics; M33 will use governed agents to propose graph updates.